## Objectives

This notebook has two objectives.

1. For each of 14 arithmetic hyperbolic surfaces, certify the length of the
   shortest closed geodesic, or systole.
2. For one genus-eight surface, use the systole certificate to prove
   $\lambda_1>1/4$ for the first nonzero Laplace eigenvalue.

For a hyperbolic group element $\gamma$,

$$\ell(\gamma)=2\operatorname{arccosh}\left(\frac{|\operatorname{tr}\gamma|}{2}\right).$$

Since $\operatorname{arccosh}$ is increasing, the shortest closed geodesic is
determined by the smallest admissible absolute trace greater than $2$.

The geometric argument therefore has two parts:

- construct a group element with trace $T$;
- prove that no smaller admissible trace occurs.

The first gives an upper bound for the systole. The second gives the matching
lower bound.

## Setup

This notebook requiresn `python-flint` and
`matplotlib`. If needed, install them with


In [ ]:
!pip install python-flint matplotlib

The next cell creates the `results/` and `images/` folders.

In [ ]:
from pathlib import Path
import csv
import json

ROOT = Path.cwd()
RESULTS = ROOT / "results"
IMAGES = ROOT / "images"
RESULTS.mkdir(exist_ok=True)
IMAGES.mkdir(exist_ok=True)

## Arithmetic in $\mathbb Z[\sqrt2]$

We will be working in the quadratic integers $a+b\sqrt2$ with $a,b\in\mathbb Z$.  We store it as
the pair `(a,b)`.  Addition is pairwise, while

$$
(a+b\sqrt2)(c+d\sqrt2)=(ac+2bd)+(ad+bc)\sqrt2.
$$

The first code cell implements this representation and basic
arithmetic operations.

In [ ]:
# We represent a + b*sqrt(2) by the pair (a, b).
def qi(a=0, b=0):
    """Construct a quadratic integer in Z[sqrt(2)].

    Args:
        a: Coefficient of 1.
        b: Coefficient of sqrt(2).

    Returns:
        The pair (a, b) representing a + b*sqrt(2).
    """
    return (int(a), int(b))


def as_qi(x):
    """Convert a scalar to the quadratic-integer representation.

    Args:
        x: A quadratic-integer pair or an integer scalar.

    Returns:
        A quadratic-integer pair.
    """
    if isinstance(x, tuple):
        return x
    return qi(x, 0)


def q_add(x, y):
    """Add two quadratic integers.

    Args:
        x: First quadratic integer.
        y: Second quadratic integer.

    Returns:
        The sum x + y in pair representation.
    """
    y = as_qi(y)
    return qi(x[0] + y[0], x[1] + y[1])


def q_subtract(x, y):
    """Subtract two quadratic integers.

    Args:
        x: First quadratic integer.
        y: Second quadratic integer.

    Returns:
        The difference x - y in pair representation.
    """
    y = as_qi(y)
    return qi(x[0] - y[0], x[1] - y[1])


def q_negative(x):
    """Negate a quadratic integer.

    Args:
        x: Quadratic integer to negate.

    Returns:
        The additive inverse of x.
    """
    return qi(-x[0], -x[1])


def q_multiply(x, y):
    """Multiply two quadratic integers.

    Args:
        x: First quadratic integer.
        y: Second quadratic integer.

    Returns:
        The product x*y in pair representation.
    """
    y = as_qi(y)
    a = x[0] * y[0] + 2 * x[1] * y[1]
    b = x[0] * y[1] + x[1] * y[0]
    return qi(a, b)


The field $\mathbb Q(\sqrt2)$ has two real embeddings:

$$\sigma_1(a+b\sqrt2)=a+b\sqrt2,\qquad\sigma_2(a+b\sqrt2)=a-b\sqrt2.$$

Conjugation exchanges them. Their product is a norm: $$N(a+b\sqrt2)=a^2-2b^2.$$

We'll see later that $\sigma_1$ describes hyperbolic motion while
$\sigma_2$ supplies the bound that makes the search finite.

In [ ]:
def q_conjugate(x):
    """Conjugates the real embedding of a quadratic integer.

    Args:
        x: Quadratic integer a + b*sqrt(2).

    Returns:
        The conjugate a - b*sqrt(2).
    """
    return qi(x[0], -x[1])


def q_norm(x):
    """Compute the field norm of a quadratic integer.

    Args:
        x: Quadratic integer a + b*sqrt(2).

    Returns:
        The integer a^2 - 2*b^2.
    """
    return x[0] * x[0] - 2 * x[1] * x[1]


We must decide inequalities such as $a+b\sqrt2>2$ without rounding $\sqrt2$. If the rational and irrational terms have opposite signs, we compare their squares. Because $\sqrt2$ is irrational, equality cannot occur accidentally.

In [ ]:
def q_sign(x, embedding=1):
    """Determine the sign of a quadratic integer exactly.

    Args:
        x: Quadratic integer to evaluate.
        embedding: Real embedding, using 1 for sqrt(2) and -1 for -sqrt(2).

    Returns:
        -1, 0, or 1 according to the sign at the selected embedding.
    """
    a = x[0]
    b = embedding * x[1]

    if a == 0:
        return (b > 0) - (b < 0)
    if b == 0 or (a > 0) == (b > 0):
        return (a > 0) - (a < 0)

    # The two terms have opposite signs, so compare their squared magnitudes.
    difference_of_squares = a * a - 2 * b * b
    if difference_of_squares > 0:
        return (a > 0) - (a < 0)
    return (b > 0) - (b < 0)


def q_compare(x, y, embedding=1):
    """Compare two quadratic integers at a real embedding.

    Args:
        x: First quadratic integer.
        y: Second quadratic integer or integer scalar.
        embedding: Real embedding, using 1 for sqrt(2) and -1 for -sqrt(2).

    Returns:
        -1 if x < y, 0 if x == y, and 1 if x > y.
    """
    return q_sign(q_subtract(x, y), embedding)

### Divisibility by a principal ideal generator

For a principal ideal $(I)$, $I$ divides $x$ if $x/I\in\mathbb Z[\sqrt2]$. Rationalizing gives

$$\frac{x}{I}=\frac{x\overline I}{N(I)},$$

so divisibility reduces to two ordinary integer remainder checks.


In [ ]:
def q_divides(divisor, x):
    """Test divisibility in Z[sqrt(2)].

    Args:
        divisor: Proposed divisor.
        x: Quadratic integer to test.

    Returns:
        True if x/divisor lies in Z[sqrt(2)], and False otherwise.
    """
    norm = q_norm(divisor)
    if norm == 0:
        return x == ZERO
    numerator = q_multiply(x, q_conjugate(divisor))
    return numerator[0] % norm == 0 and numerator[1] % norm == 0

The next cell defines a function that converts a quadratic integer into a JSON format for computer certification.

In [ ]:
def q_json(x):
    """Convert a quadratic integer to a JSON-serializable dictionary.

    Args:
        x: Quadratic integer in pair representation.

    Returns:
        A dictionary containing the coefficients of 1 and sqrt(2).
    """
    return {"a": x[0], "b": x[1]}

Next we will store some commonly used constants for convenience.

In [ ]:
ZERO = qi(0, 0)
ONE = qi(1, 0)
SQRT2 = qi(0, 1)

### Testing
The following assertions test the norm, multiplication, and comparison operations on a simple example as a sanity check.

In [ ]:
q = qi(3, 2)
assert q_norm(q) == 1
assert q_multiply(q, q_conjugate(q)) == ONE
assert q_multiply(qi(1, 1), qi(1, 1)) == qi(3, 2)
assert q_compare(qi(9, 6), 2) > 0
print("basic Z[sqrt(2)] arithmetic: OK")
